In [1]:
import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace

load_dotenv("/home/jovyan/work/.env")

MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("DB_NAME")
PROCESSED_COLLECTION = os.getenv("PROCESSED_COLLECTION")

spark = (
    SparkSession.builder
    .appName("Exportar_Datos_Dashboard_AutoTec")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

df = (
    spark.read.format("mongodb")
    .option("database", DB_NAME)
    .option("collection", PROCESSED_COLLECTION)
    .load()
)

df_clean = df.select(
    "marca",
    "modelo",
    "year",
    "precio",
    "kilometraje",
    "combustible",
    "ciudad",
    "url"
).dropDuplicates(["url"])

df_clean = df_clean.withColumn(
    "precio_num",
    regexp_replace(col("precio"), "[^0-9]", "").cast("double")
)

df_clean = df_clean.withColumn(
    "km_num",
    regexp_replace(col("kilometraje"), "[^0-9]", "").cast("double")
)

df_clean = df_clean.withColumn(
    "year_limpio",
    regexp_replace(col("year"), "[^0-9]", "").cast("int")
)

df_clean = df_clean.filter(col("precio_num").isNotNull())
df_clean = df_clean.filter(col("km_num").isNotNull())
df_clean = df_clean.filter(col("year_limpio").isNotNull())
df_clean = df_clean.filter(col("precio_num") > 0)
df_clean = df_clean.filter(col("km_num") >= 0)

ruta_csv = "/home/jovyan/work/autotec/Jocelyn/datos_autotec_dashboard.csv"

df_clean.toPandas().to_csv(ruta_csv, index=False)

print("CSV creado correctamente en:", ruta_csv)
print("Registros exportados:", df_clean.count())

CSV creado correctamente en: /home/jovyan/work/autotec/Jocelyn/datos_autotec_dashboard.csv
Registros exportados: 1988
